In [2]:
import os
from pyannote.audio import Pipeline
from pydub import AudioSegment
import whisper


In [3]:


# Ruta del archivo de audio largo
INPUT_AUDIO = "./SpeechT5/SWITCH2Fedelobo.wav"
OUTPUT_DIR = "./SpeechT5"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [8]:

# 1. Ejecutar VAD con pyannote.audio
print("Detectando segmentos de voz...")
pipeline = Pipeline.from_pretrained("pyannote/voice-activity-detection",use_auth_token="miTOKEN_HF")
vad_result = pipeline(INPUT_AUDIO)


# 2. Cargar audio para segmentar
audio = AudioSegment.from_wav(INPUT_AUDIO)


In [10]:

# 3. Segmentación: guardar cada segmento de voz como un archivo WAV
segment_paths = []
for i, (segment, _) in enumerate(vad_result.itertracks(yield_label=False)):
    start_ms = int(segment.start * 1000)
    end_ms = int(segment.end * 1000)
    segment_audio = audio[start_ms:end_ms]
    
    if len(segment_audio) > 1000:
        segment_path = os.path.join(OUTPUT_DIR, f"{i:03d}.wav")
        segment_audio.export(segment_path, format="wav")
        segment_paths.append(segment_path)

print(f"Se generaron {len(segment_paths)} segmentos.")


Se generaron 5 segmentos.


In [11]:

# 4. Transcripción con Whisper
print("Transcribiendo segmentos con Whisper...")
model = whisper.load_model("small")  # Usa "base" si tienes recursos limitados
metadata = []

for i, path in enumerate(segment_paths):
    result = model.transcribe(path, language="es")
    text = result["text"].strip()
    
    if text:  # Solo incluir segmentos con texto no vacío
        filename = os.path.basename(path)
        id_ = os.path.splitext(filename)[0]
        metadata.append(f"{id_}|{text}")
        print(f"[{id_}] {text}")


Transcribiendo segmentos con Whisper...


100%|███████████████████████████████████████| 461M/461M [00:08<00:00, 55.1MiB/s]


[000] Sí, mi experiencia con la Nintendo Switch 2, señores y señores, ya les puedo contar como fue o qué sucedió esta semana. Hace como unos dos meses Nintendo me contacta y me dice, no, oye, vamos a anunciar o vamos a llevar a ciertas personas a probar la Nintendo Switch 2. Jalas o pandeas, me dijeron. Así decía el correo. Jalas o pandeas. La Sushi 2. Vale, madre, es que sí, dije hace mucho no voy a eventos, hace mucho no voy a probar ciertas cosas y dije, güey, sí me gustaría estar. Sí me gustaría estar. Recuerdo que el lanzamiento de la primera Nintendo Switch fue en Nueva York y yo sí le tuve un chingo de envidia a la gente que fue a probarla. Entonces pues ya fui, me tocó ir a Nueva York, gente. Me tocó ir a Nueva York. La neta para mí pues sí fue una toda una experiencia, un sueño cumplido, porque me pongo muy contento cada vez que una compañía me toma en cuenta para probar los primeros minutos de vida de una consola. ¿Cuál fue la primera? Sí, yo creo que la primera fue Xbox. La 

In [13]:

# 5. Guardar metadata.csv
with open(os.path.join(OUTPUT_DIR, "metadata.csv"), "w", encoding="utf-8") as f:
    f.write("\n".join(metadata))

print("\n✅ Dataset listo en:", OUTPUT_DIR)


✅ Dataset listo en: ./SpeechT5
